# R09 EEG–握力特征池展示

本笔记本从团队数据接口读取 R09，使用 `expanded_multiscale` 配方生成因果 EEG 特征池，并展示频带功率、时域、谱形、burst、标签、时间窗和通道分布。

In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from grip_data_interface import load_flight_trials
from grip_feature_pool import build_grip_feature_pool

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
})
BLUE = '#2F6B9A'
GOLD = '#C6922F'
INK = '#252A30'
GRID = '#D9DEE3'

DATA_DIR = Path('0807华山grip flight')
EEG_PATH = DATA_DIR / 'testS001R09.dat.larkcache'
TASK_PATH = DATA_DIR / 'testS001R09_1.dat'
RECIPE = 'expanded_multiscale'
WINDOW_MS = 500
STEP_MS = 50

assert EEG_PATH.exists(), EEG_PATH
assert TASK_PATH.exists(), TASK_PATH

In [ ]:
started = perf_counter()
r09_trials = load_flight_trials(
    EEG_PATH,
    TASK_PATH,
    segment='trial',
)
pool = build_grip_feature_pool(
    r09_trials,
    recipe=RECIPE,
    window_ms=WINDOW_MS,
    step_ms=STEP_MS,
    causal=True,
    reference='none',
)
elapsed_s = perf_counter() - started
pool.validate()

## tl;dr

In [ ]:
n_windows, n_channels, n_features = pool.X.shape
flight_windows = int(pool.windows['mask_flight'].sum())
summary = pd.DataFrame({
    'item': ['recording', 'trials', 'windows', 'flight windows', 'channels', 'features', 'feature array', 'build time'],
    'value': [
        pool.manifest['recording_ids'][0],
        len(r09_trials['meta']),
        f'{n_windows:,}',
        f'{flight_windows:,}',
        n_channels,
        n_features,
        f'{n_windows} × {n_channels} × {n_features}',
        f'{elapsed_s:.1f} s',
    ],
})
display(summary.style.hide(axis='index'))
display(Markdown(
    f'R09 共生成 **{n_windows:,}** 个历史窗口，每个窗口包含 **{n_channels}** 个通道和 '
    f'**{n_features}** 个多尺度特征；其中 **{flight_windows:,}** 个窗口落在 flight 阶段。'
))

## Context & Methods

### Key assumptions

- 上游使用 `segment='trial'`，使 flight 开始处的窗口保留 Countdown 历史。
- 每个特征窗口只使用标签时刻及之前的 EEG；滤波使用因果模式。
- 频带名称复现文献定义，但这不是对每篇论文全部预处理细节的逐项复现。
- 不同特征共享同一标签时刻，但使用250–2000 ms不等的历史窗口；具体长度见特征目录。
- GamePhase、Collision 和 outcome 仅作为标签或筛选条件，不进入 EEG 特征。

In [ ]:
method_summary = pd.Series({
    'source EEG': EEG_PATH.name,
    'source task': TASK_PATH.name,
    'source interface version': pool.manifest['source_interface_versions'][0],
    'feature interface version': pool.manifest['feature_interface_version'],
    'sampling rate (Hz)': pool.manifest['sampling_rate_hz'],
    'recipe': pool.manifest['recipe'],
    'default feature window (ms)': WINDOW_MS,
    'output step (ms)': STEP_MS,
    'maximum history (ms)': pool.manifest['history_window_ms'],
    'label lag (ms)': pool.manifest['label_lag_ms'],
    'label anchor': pool.manifest['label_anchor'],
    'causal filtering': pool.manifest['causal'],
    'reference': pool.manifest['reference'],
    'notches (Hz)': ', '.join(map(str, pool.manifest['notch_hz'])),
    'filter order': pool.manifest['filter_order'],
    'log band power': pool.manifest['log_power'],
    'burst threshold': f"median + {pool.manifest['burst_threshold_mad']:g} × robust SD",
    'burst baseline': pool.manifest['burst_threshold_baseline'],
    'band-power unit': 'dB(source signal units²)',
}, name='value').to_frame()
display(method_summary)

### 特征池对象与数组契约

`FeaturePool` 把数值数组、标签、窗口索引和生成参数放在同一个对象中。下表给出这些对象之间的对应关系。

In [ ]:
object_contract = pd.DataFrame([
    {'object': 'pool.X', 'shape / type': str(pool.X.shape), 'role': 'window × EEG channel × feature 数值张量'},
    {'object': 'pool.labels', 'shape / type': str(pool.labels.shape), 'role': '每个 window 的预测目标和任务标签'},
    {'object': 'pool.windows', 'shape / type': str(pool.windows.shape), 'role': '每个 window 的 recording、trial、时间和 mask'},
    {'object': 'pool.feature_info', 'shape / type': 'dict', 'role': 'feature/channel 轴的名称、单位和参数'},
    {'object': 'pool.manifest', 'shape / type': 'dict', 'role': '配方、滤波、窗口、版本和生成警告'},
    {'object': 'pool.raw_windows', 'shape / type': str(pool.raw_windows), 'role': '本次未保存；仅 include_raw_windows=True 时存在'},
])
display(object_contract.style.hide(axis='index'))
assert pool.X.shape[0] == len(pool.labels) == len(pool.windows)

## Data

### Trial 与完整特征目录

每一项特征都对每个EEG通道分别计算。`window_ms` 是该特征实际使用的历史长度，不一定等于默认500 ms。

In [ ]:
trial_columns = ['trial_key', 'outcome', 'collision', 'duration_s', 'n_samples', 'force_mean', 'force_std', 'force_max']
display(r09_trials['meta'][trial_columns].style.format({
    'duration_s': '{:.2f}', 'force_mean': '{:.4f}', 'force_std': '{:.4f}', 'force_max': '{:.4f}'
}))

feature_descriptions = {
    'lmp': '窗口平均电位；描述慢电位变化',
    'slope': '窗口内最小二乘直线斜率',
    'rms': '均方根幅度',
    'line_length': '相邻采样绝对变化率的窗口均值',
    'hjorth_activity': '信号方差',
    'hjorth_mobility': '一阶导数方差与原信号方差之比的平方根',
    'hjorth_complexity': '一阶导数mobility与原信号mobility之比',
    'spectral_entropy': '七个基础频带功率分布的归一化熵；0集中、1平坦',
    'spectral_centroid': '七个基础频带的功率加权中心频率',
}
feature_catalog = pd.DataFrame(pool.feature_info['feature_axis'])
feature_catalog['description'] = feature_catalog.apply(
    lambda row: (
        feature_descriptions.get(row['name'])
        or ('频带滤波后平方，在历史窗内求均值并转为dB' if row['family'] == 'band_power' else None)
        or ({
            'occupancy': '功率包络超过阈值的时间比例',
            'rate': '窗口内burst数量/秒',
            'mean_duration': '窗口内burst平均持续时间',
        }[row['name'].rsplit('_', 1)[-1]] if row['family'] == 'burst' and not row['name'].endswith('mean_duration') else None)
        or ('窗口内burst平均持续时间' if row['name'].endswith('mean_duration') else '—')
    ),
    axis=1,
)
feature_catalog['sources'] = feature_catalog['recipe_sources'].apply(', '.join)
feature_catalog['frequency_hz'] = feature_catalog.apply(
    lambda row: (f"{row['low_hz']:g}–{row['high_hz']:g}" if pd.notna(row.get('low_hz')) else '—'),
    axis=1,
)
feature_catalog['extra_parameters'] = feature_catalog.apply(
    lambda row: (f"envelope={row['envelope_ms']:g} ms" if pd.notna(row.get('envelope_ms')) else '—'),
    axis=1,
)
display(feature_catalog[[
    'index', 'name', 'family', 'description', 'window_ms', 'frequency_hz',
    'unit', 'extra_parameters', 'sources',
]].fillna('—').style.hide(axis='index'))
display(
    feature_catalog.groupby(['family', 'window_ms']).size()
    .rename('n_features').reset_index().style.hide(axis='index')
)

feature_distribution = pd.DataFrame([
    {
        'feature': feature_name,
        'min': float(np.nanmin(pool.X[:, :, feature_i])),
        'median': float(np.nanmedian(pool.X[:, :, feature_i])),
        'p99': float(np.nanpercentile(pool.X[:, :, feature_i], 99)),
        'max': float(np.nanmax(pool.X[:, :, feature_i])),
        'zero_fraction': float(np.mean(pool.X[:, :, feature_i] == 0)),
    }
    for feature_i, feature_name in enumerate(pool.feature_names)
])
display(feature_distribution.style.hide(axis='index').format({
    'min': '{:.4g}', 'median': '{:.4g}', 'p99': '{:.4g}',
    'max': '{:.4g}', 'zero_fraction': '{:.3f}',
}))

### 标签、窗口字段与掩码字典

标签用于预测或评价，窗口字段用于追溯和划分。GamePhase、Collision和outcome不会进入EEG特征张量。

In [ ]:
label_definitions = {
    'window_id': '与pool.X第一维完全一致的连续编号',
    'force_raw': '标签时刻的原始握力',
    'force_normalized': '任务保存的0–1归一化握力',
    'force_derivative': '握力一阶时间导数 dF/dt',
    'force_acceleration': '握力二阶时间导数 d²F/dt²',
    'is_force_active': '归一化握力是否超过活动阈值',
    'game_phase': '标签时刻的GamePhase整数状态',
    'collision': '标签时刻Collision状态是否为正',
    'collision_onset': '自上个输出步长以来是否出现Collision onset',
    'outcome': '完整Trial结果：success/failure/incomplete',
    'trial_collision': '完整Trial内是否曾发生碰撞',
}
label_dictionary = pd.DataFrame([
    {'field': name, 'dtype': str(pool.labels[name].dtype), 'description': label_definitions[name]}
    for name in pool.labels.columns
])
display(label_dictionary.style.hide(axis='index'))

window_definitions = {
    'window_id': '与pool.X和pool.labels对应的行号',
    'recording_id': '记录编号，本笔记本为R09',
    'trial_index0': '从0开始的原始Trial索引',
    'trial_id': '从1开始的Trial ID',
    'trial_key': 'recording与Trial组成的全局分组键',
    'window_start_sample': '最大历史窗口在trial内的起始EEG采样点',
    'window_stop_sample': '最大历史窗口的右开区间终点',
    'label_sample': '标签在trial内的EEG采样点',
    'global_eeg_label_sample': '标签在完整EEG记录中的采样点',
    'window_start_s': '最大历史窗口的trial相对起始时间',
    'window_end_s': '最大历史窗口结束时间',
    'label_time_s': '标签的trial相对时间',
    'source_clock_ms': '标签时刻相对trial起点的SourceTime时钟',
    'outcome': '完整Trial结果',
    'trial_collision': '完整Trial是否碰撞',
    'game_phase_label': '标签时刻GamePhase文本',
    'mask_flight': 'GamePhase为Playing或Hit',
    'mask_playing': 'GamePhase严格等于Playing',
    'mask_force_active': '标签时刻握力超过活动阈值',
}
window_dictionary = pd.DataFrame([
    {'field': name, 'dtype': str(pool.windows[name].dtype), 'description': window_definitions[name]}
    for name in pool.windows.columns
])
display(window_dictionary.style.hide(axis='index'))

game_phase_table = pd.DataFrame([
    {'game_phase': code, 'label': label}
    for code, label in sorted(r09_trials['game_phase_labels'].items())
])
display(game_phase_table.style.hide(axis='index'))

### 通道轴

当前没有电极植入坐标或脑区信息，因此只保留接口通道名称和原始通道索引，不解释空间耦合。

In [ ]:
channel_catalog = pd.DataFrame(pool.feature_info['channel_axis'])
display(channel_catalog.head(20).style.hide(axis='index'))
print(f"Showing 20 of {len(channel_catalog)} channels; all channel metadata remain in pool.feature_info['channel_axis'].")

### Manifest：本次特征池怎样生成

下面完整展示本次运行的可序列化生成参数；保存特征池时这些内容会写入 `manifest.json`。

In [ ]:
manifest_table = pd.DataFrame([
    {
        'field': key,
        'value': (', '.join(map(str, value)) if isinstance(value, list) else str(value)),
    }
    for key, value in pool.manifest.items()
])
display(manifest_table.style.hide(axis='index'))

### 五文件导出契约

`pool.save(output_dir)` 将内存对象保存为下列五个文件。Parquet导出需要安装 `pyarrow` 或 `fastparquet`。

In [ ]:
export_contract = pd.DataFrame([
    {'file': 'features.npz', 'content': 'X；可选raw_windows', 'row/axis relation': 'X[window_id, channel, feature]'},
    {'file': 'labels.parquet', 'content': '连续目标和任务标签', 'row/axis relation': '每行对应一个window_id'},
    {'file': 'windows.parquet', 'content': 'trial、时间、GamePhase与mask', 'row/axis relation': '每行对应一个window_id'},
    {'file': 'feature_names.json', 'content': 'feature_axis和channel_axis', 'row/axis relation': '解释X的后两个轴'},
    {'file': 'manifest.json', 'content': '版本、配方、滤波、窗口和阈值', 'row/axis relation': '解释整套数据怎样生成'},
])
display(export_contract.style.hide(axis='index'))

### 窗口与标签预览

`window_id` 是 `pool.X` 第一维的行号；每一行都能追溯到 recording、trial 和标签时刻。

In [ ]:
preview_columns = [
    'window_id', 'trial_key', 'window_start_s', 'window_end_s', 'label_time_s',
    'game_phase_label', 'mask_flight', 'force_normalized', 'force_derivative',
    'collision_onset', 'outcome',
]
window_preview = pool.windows.merge(
    pool.labels.drop(columns=['outcome', 'trial_collision']),
    on='window_id',
    validate='one_to_one',
)
display(window_preview[preview_columns].head(10).style.format({
    'window_start_s': '{:.3f}', 'window_end_s': '{:.3f}', 'label_time_s': '{:.3f}',
    'force_normalized': '{:.4f}', 'force_derivative': '{:.4f}',
}))

## Results

### 1. 全部多尺度特征的时间概览

先对通道取中位数，再对每个特征沿时间做 robust z-score。颜色表示相对变化，不比较不同频带的绝对 dB 数值。

In [ ]:
def robust_z(values, axis=0):
    values = np.asarray(values, dtype=float)
    median = np.nanmedian(values, axis=axis, keepdims=True)
    mad = np.nanmedian(np.abs(values - median), axis=axis, keepdims=True)
    scale = np.where(mad > 0, 1.4826 * mad, np.nanstd(values, axis=axis, keepdims=True))
    return (values - median) / np.where(scale > 0, scale, 1.0)

time_feature = np.nanmedian(pool.X, axis=1)
time_feature_z = np.clip(robust_z(time_feature, axis=0), -3, 3)
trial_boundaries = np.flatnonzero(pool.windows['trial_key'].to_numpy()[1:] != pool.windows['trial_key'].to_numpy()[:-1]) + 1

fig, ax = plt.subplots(figsize=(15, 7))
image = ax.imshow(time_feature_z.T, aspect='auto', interpolation='nearest', cmap='RdBu_r', vmin=-3, vmax=3)
for boundary in trial_boundaries:
    ax.axvline(boundary - 0.5, color=INK, lw=0.8, alpha=0.7)
ax.set_yticks(np.arange(n_features))
ax.set_yticklabels(pool.feature_names, fontsize=8)
ax.set_xlabel('Sequential feature windows (trial boundaries in black)')
ax.set_ylabel('Feature')
ax.set_title('R09 expanded multiscale feature pool over time')
colorbar = fig.colorbar(image, ax=ax, pad=0.01)
colorbar.set_label('Robust z-score across windows')
fig.tight_layout()
plt.show()

### 2. 通道—特征分布

对 flight 窗口取时间中位数，再在每个特征内跨通道标准化，用于定位相对高/低功率通道。

In [ ]:
flight_mask = pool.windows['mask_flight'].to_numpy(dtype=bool)
channel_feature = np.nanmedian(pool.X[flight_mask], axis=0)
channel_feature_z = np.clip(robust_z(channel_feature, axis=0), -3, 3)

fig, ax = plt.subplots(figsize=(15, 10))
image = ax.imshow(channel_feature_z, aspect='auto', interpolation='nearest', cmap='RdBu_r', vmin=-3, vmax=3)
ax.set_xticks(np.arange(n_features))
ax.set_xticklabels(pool.feature_names, rotation=60, ha='right', fontsize=8)
channel_tick_step = max(1, n_channels // 20)
channel_ticks = np.arange(0, n_channels, channel_tick_step)
ax.set_yticks(channel_ticks)
ax.set_yticklabels(np.asarray(pool.channel_names)[channel_ticks], fontsize=8)
ax.set_xlabel('Feature')
ax.set_ylabel('EEG channel')
ax.set_title('R09 channel × feature profile during flight')
colorbar = fig.colorbar(image, ax=ax, pad=0.01)
colorbar.set_label('Robust z-score across channels')
fig.tight_layout()
plt.show()

### 3. 单个 Trial：特征与握力时间关系

展示 Trial 1 的跨通道中位特征。各曲线分别标准化，因此只比较时间形态，不比较原始幅值。

In [ ]:
selected_features = [
    'lmp', 'slope', 'rms', 'bandpower_70_115Hz',
    'burst_beta_occupancy', 'burst_high_gamma_occupancy', 'spectral_entropy',
]
first_trial = pool.windows['trial_key'].iloc[0]
trial_mask = (pool.windows['trial_key'] == first_trial).to_numpy()
trial_time = pool.windows.loc[trial_mask, 'label_time_s'].to_numpy()
trial_force = pool.labels.loc[trial_mask, 'force_normalized'].to_numpy()
trial_feature = np.nanmedian(pool.X[trial_mask], axis=1)
force_scale = np.nanstd(trial_force)
force_z = np.clip(
    (trial_force - np.nanmean(trial_force)) / (force_scale if force_scale > 0 else 1.0),
    -4, 4,
)

fig, axes = plt.subplots(len(selected_features), 1, figsize=(13, 14), sharex=True)
for ax, feature_name in zip(axes, selected_features):
    feature_index = pool.feature_names.index(feature_name)
    feature_z = np.clip(robust_z(trial_feature[:, feature_index], axis=0), -4, 4)
    ax.plot(trial_time, force_z, color=INK, lw=1.5, label='Force (z)')
    ax.plot(trial_time, feature_z, color=BLUE, lw=1.0, alpha=0.85, label=feature_name)
    ax.axhline(0, color=GRID, lw=0.8)
    ax.set_ylabel('z')
    ax.set_title(feature_name, loc='left', fontsize=10)
axes[0].legend(frameon=False, ncol=2, loc='upper right')
axes[-1].set_xlabel('Trial-relative label time (s)')
fig.suptitle(f'{first_trial}: force and channel-median EEG features', y=1.01, fontweight='bold')
fig.tight_layout()
plt.show()

### 4. Trial 1：全部特征与握力的时间关系

按特征族分图展示全部 39 项特征。每项特征先跨通道取中位数，再在 Trial 1 内做稳健标准化；握力做普通标准化。曲线裁剪到 ±4，只用于比较同步变化、领先或滞后等时间形态，不表示相关性显著或模型性能。

In [ ]:
family_order = ['time_domain', 'band_power', 'spectral_shape', 'burst']
family_titles = {
    'time_domain': 'Time-domain features',
    'band_power': 'Band-power features',
    'spectral_shape': 'Spectral-shape features',
    'burst': 'Burst features',
}
plotted_features = []

for family in family_order:
    family_rows = feature_catalog.loc[
        feature_catalog['family'].eq(family), ['name', 'window_ms']
    ].reset_index(drop=True)
    family_features = family_rows['name'].tolist()
    plotted_features.extend(family_features)

    n_cols = 2 if len(family_features) > 12 else 1
    n_rows = int(np.ceil(len(family_features) / n_cols))
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(18 if n_cols == 2 else 13, 2.0 * n_rows + 1.5),
        sharex='col', squeeze=False,
    )
    axes_flat = axes.ravel()

    for ax, row in zip(axes_flat, family_rows.itertuples(index=False)):
        feature_index = pool.feature_names.index(row.name)
        feature_z = np.clip(robust_z(trial_feature[:, feature_index], axis=0), -4, 4)
        ax.plot(trial_time, force_z, color=INK, lw=1.35, alpha=0.8, label='Force (z)')
        ax.plot(trial_time, feature_z, color=BLUE, lw=1.0, alpha=0.85, label='Feature (robust z)')
        ax.axhline(0, color=GRID, lw=0.8)
        ax.set_ylim(-4.2, 4.2)
        ax.set_ylabel('z')
        ax.set_title(f'{row.name}  ·  {row.window_ms:g} ms', loc='left', fontsize=9)

    for ax in axes_flat[len(family_features):]:
        ax.set_visible(False)
    axes_flat[0].legend(frameon=False, ncol=2, loc='upper right')
    for col in range(n_cols):
        visible_axes = [axes[row, col] for row in range(n_rows) if axes[row, col].get_visible()]
        if visible_axes:
            visible_axes[-1].set_xlabel('Trial-relative label time (s)')
    fig.suptitle(
        f'{first_trial}: {family_titles[family]} and force',
        y=1.005, fontweight='bold',
    )
    fig.tight_layout()
    plt.show()

assert len(plotted_features) == len(pool.feature_names)
assert set(plotted_features) == set(pool.feature_names)
print(f'Displayed {len(plotted_features)} / {len(pool.feature_names)} features for {first_trial}.')

### 5. High-gamma 与握力相关性最高的通道

这是描述性筛查，不是交叉验证后的建模结果。通道排名不能直接当作可泛化的通道选择。

In [ ]:
high_gamma_name = 'bandpower_70_115Hz'
high_gamma_index = pool.feature_names.index(high_gamma_name)
flight_force = pool.labels.loc[flight_mask, 'force_normalized'].to_numpy()
channel_correlations = np.asarray([
    pd.Series(pool.X[flight_mask, channel_i, high_gamma_index]).corr(
        pd.Series(flight_force), method='spearman'
    )
    for channel_i in range(n_channels)
])
top_indices = np.argsort(np.nan_to_num(np.abs(channel_correlations), nan=-1))[-15:]
top_table = pd.DataFrame({
    'channel': np.asarray(pool.channel_names)[top_indices],
    'spearman_r': channel_correlations[top_indices],
}).sort_values('spearman_r')

fig, ax = plt.subplots(figsize=(8, 6))
colors = [GOLD if value >= 0 else BLUE for value in top_table['spearman_r']]
ax.barh(top_table['channel'], top_table['spearman_r'], color=colors, edgecolor=INK, linewidth=0.4)
ax.axvline(0, color=INK, lw=0.8)
ax.set_xlabel('Spearman r with normalized force')
ax.set_ylabel('EEG channel')
ax.set_title(f'R09 top |correlation| channels: {high_gamma_name}')
ax.grid(axis='x', color=GRID, linewidth=0.6)
fig.tight_layout()
plt.show()
display(top_table.sort_values('spearman_r', ascending=False).reset_index(drop=True).style.format({'spearman_r': '{:.3f}'}))

### 6. 每个特征 × 通道 × Trial 的滞后 Spearman 相关

在每个 Trial 的 **flight 窗口**内计算相关，扫描 −1500～+1500 ms，步长与特征池一致（50 ms）。正滞后表示 EEG 特征领先握力，例如 `peak_lag_ms = +250` 表示比较 `feature(t)` 与 `force(t + 250 ms)`。

`lagged_spearman_detail` 保存每个特征 × 通道 × Trial 的四项指标。`feature_spearman_report` 先在所有通道和 Trial 间对每个滞后的相关系数取中位数，再按特征报告 `rho_lag0`、`peak_abs_rho`、`peak_rho` 和 `peak_lag_ms`，减少单个通道或单个 Trial 极值的影响。每个滞后都在实际重叠片段内重新计算秩，因此都是标准 Spearman 相关。

In [ ]:
from scipy.stats import rankdata

MAX_LAG_MS = 1500
CORRELATION_MASK = 'mask_flight'
correlation_step_ms = int(pool.manifest['step_ms'])
lag_steps = np.arange(
    -MAX_LAG_MS // correlation_step_ms,
    MAX_LAG_MS // correlation_step_ms + 1,
)
lag_ms = lag_steps * correlation_step_ms
zero_lag_index = int(np.flatnonzero(lag_steps == 0)[0])

def lagged_rank_correlations(feature_values, force_values, shifts):
    """Return lag × flattened-feature correlations; positive lag means feature leads force."""
    feature_values = np.asarray(feature_values, dtype=float)
    force_values = np.asarray(force_values, dtype=float)
    if not np.isfinite(feature_values).all() or not np.isfinite(force_values).all():
        raise ValueError('Lagged Spearman input contains non-finite values')

    correlations = np.full((len(shifts), feature_values.shape[1]), np.nan, dtype=float)

    for lag_index, shift in enumerate(shifts):
        if shift > 0:
            x_values = feature_values[:-shift]
            y_values = force_values[shift:]
        elif shift < 0:
            x_values = feature_values[-shift:]
            y_values = force_values[:shift]
        else:
            x_values = feature_values
            y_values = force_values

        x_ranks = rankdata(x_values, axis=0, method='average')
        y_ranks = rankdata(y_values, method='average')
        x_centered = x_ranks - x_ranks.mean(axis=0)
        y_centered = y_ranks - y_ranks.mean()
        covariance = x_centered.T @ y_centered
        denominator = np.sqrt(
            np.square(x_centered).sum(axis=0) * np.square(y_centered).sum()
        )
        correlations[lag_index] = np.divide(
            covariance, denominator,
            out=np.full_like(covariance, np.nan),
            where=denominator > 0,
        )
    return correlations

detail_frames = []
trial_correlation_cubes = []
channel_axis = np.repeat(np.asarray(pool.channel_names), n_features)
feature_axis = np.tile(np.asarray(pool.feature_names), n_channels)

for trial_key in pool.windows['trial_key'].drop_duplicates():
    trial_mask_for_correlation = (
        pool.windows['trial_key'].eq(trial_key).to_numpy()
        & pool.windows[CORRELATION_MASK].to_numpy(dtype=bool)
    )
    trial_force_for_correlation = pool.labels.loc[
        trial_mask_for_correlation, 'force_normalized'
    ].to_numpy()
    trial_features_for_correlation = pool.X[trial_mask_for_correlation].reshape(
        trial_mask_for_correlation.sum(), -1
    )
    if len(trial_force_for_correlation) <= 2 * max(abs(lag_steps)) + 2:
        raise ValueError(f'{trial_key} is too short for ±{MAX_LAG_MS} ms lag analysis')

    trial_rho_flat = lagged_rank_correlations(
        trial_features_for_correlation, trial_force_for_correlation, lag_steps
    )
    trial_rho = trial_rho_flat.reshape(len(lag_steps), n_channels, n_features)
    trial_correlation_cubes.append(trial_rho)

    finite_peak_values = np.where(np.isfinite(trial_rho_flat), np.abs(trial_rho_flat), -np.inf)
    has_valid_peak = np.isfinite(trial_rho_flat).any(axis=0)
    peak_indices = np.argmax(finite_peak_values, axis=0)
    peak_rho = np.take_along_axis(trial_rho_flat, peak_indices[None, :], axis=0)[0]
    peak_rho[~has_valid_peak] = np.nan
    peak_lag_for_detail = lag_ms[peak_indices].astype(float)
    peak_lag_for_detail[~has_valid_peak] = np.nan
    detail_frames.append(pd.DataFrame({
        'trial_key': trial_key,
        'channel': channel_axis,
        'feature': feature_axis,
        'rho_lag0': trial_rho_flat[zero_lag_index],
        'peak_abs_rho': np.abs(peak_rho),
        'peak_rho': peak_rho,
        'peak_lag_ms': peak_lag_for_detail,
        'n_flight_windows': len(trial_force_for_correlation),
    }))

lagged_spearman_detail = pd.concat(detail_frames, ignore_index=True)
all_trial_channel_rho = np.concatenate(trial_correlation_cubes, axis=1)
feature_lagged_rho = np.nanmedian(all_trial_channel_rho, axis=1)
finite_feature_peaks = np.where(np.isfinite(feature_lagged_rho), np.abs(feature_lagged_rho), -np.inf)
valid_feature_peaks = np.isfinite(feature_lagged_rho).any(axis=0)
feature_peak_indices = np.argmax(finite_feature_peaks, axis=0)
feature_peak_rho = np.take_along_axis(
    feature_lagged_rho, feature_peak_indices[None, :], axis=0
)[0]
feature_peak_rho[~valid_feature_peaks] = np.nan
feature_peak_lag_ms = lag_ms[feature_peak_indices].astype(float)
feature_peak_lag_ms[~valid_feature_peaks] = np.nan
feature_family_map = feature_catalog.set_index('name')['family']
feature_spearman_report = pd.DataFrame({
    'feature': pool.feature_names,
    'family': [feature_family_map[name] for name in pool.feature_names],
    'rho_lag0': feature_lagged_rho[zero_lag_index],
    'peak_abs_rho': np.abs(feature_peak_rho),
    'peak_rho': feature_peak_rho,
    'peak_lag_ms': feature_peak_lag_ms,
})

display(pd.Series({
    'analysis mask': CORRELATION_MASK,
    'lag range (ms)': f'{lag_ms.min()} to {lag_ms.max()}',
    'lag step (ms)': correlation_step_ms,
    'detail rows': f'{len(lagged_spearman_detail):,}',
    'reported features': len(feature_spearman_report),
}, name='value').to_frame())
display(
    feature_spearman_report.style
    .format({
        'rho_lag0': '{:+.3f}',
        'peak_abs_rho': '{:.3f}',
        'peak_rho': '{:+.3f}',
        'peak_lag_ms': '{:+.0f}',
    })
    .background_gradient(subset=['peak_abs_rho'], cmap='Blues')
)
display(Markdown(
    '`lagged_spearman_detail` 含完整组合级结果；以下仅预览绝对峰值最高的 12 个组合。'
))
display(
    lagged_spearman_detail.nlargest(12, 'peak_abs_rho').style
    .format({
        'rho_lag0': '{:+.3f}',
        'peak_abs_rho': '{:.3f}',
        'peak_rho': '{:+.3f}',
        'peak_lag_ms': '{:+.0f}',
    })
)

## Checks

检查特征、标签、窗口索引是否一一对应，并演示下游接口。

In [ ]:
checks = {
    'X / labels / windows row alignment': len(pool.labels) == len(pool.windows) == len(pool.X),
    'feature values finite': bool(np.isfinite(pool.X).all()),
    'window_id sequential': bool(np.array_equal(pool.windows['window_id'], np.arange(len(pool.windows)))),
    'channel metadata aligned': len(pool.channel_names) == pool.X.shape[1],
    'feature metadata aligned': len(pool.feature_names) == pool.X.shape[2],
    'all trials represented': pool.windows['trial_key'].nunique() == len(r09_trials['meta']),
    'lagged Spearman detail complete': len(lagged_spearman_detail) == pool.windows['trial_key'].nunique() * n_channels * n_features,
    'feature Spearman report complete': len(feature_spearman_report) == n_features,
}
check_table = pd.DataFrame({'check': checks.keys(), 'passed': checks.values()})
display(check_table.style.hide(axis='index'))
assert all(checks.values())

In [ ]:
X, y, groups = pool.as_sklearn(
    target='force_normalized',
    mask='mask_flight',
)
downstream_summary = pd.Series({
    'X shape': str(X.shape),
    'y shape': str(y.shape),
    'unique trial groups': len(np.unique(groups)),
    'group examples': ', '.join(map(str, np.unique(groups)[:3])),
}, name='value').to_frame()
display(downstream_summary)

## Takeaways

- `pool.X` 保留 `window × channel × feature` 结构，可对频带、时域、谱形和burst家族做消融。
- `expanded_multiscale` 用250–2000 ms不同历史窗估计39个特征，但全部对齐到同一标签时刻。
- `pool.labels` 保存连续握力、握力变化率、GamePhase、Collision onset 和 outcome。
- `pool.windows` 负责把每一行追溯到 recording、trial 和时间，并提供 flight/playing mask。
- 建模时使用 `groups=trial_key` 做完整 trial 划分；不要随机拆分重叠窗口。
- high-gamma 通道相关图仅用于描述，不应替代训练集内的通道选择。
- 滞后 Spearman 表在 flight 阶段按 Trial 和通道计算，并用跨通道、跨 Trial 中位相关生成39项特征报告；重叠窗口意味着这些数值不能直接配普通独立样本 p 值。
- 每个 trial 开头的低频特征可能包含因果滤波 settling；正式建模可增加 warm-up mask 或提供更长的 trial 前历史。

## Optional export

安装 `pyarrow` 后可取消下面代码的注释，写出五文件特征数据集。生成目录属于患者派生数据，不应提交到 Git。

In [ ]:
# pool.save('generated_features/R09_expanded_multiscale_v1')